# 21 — Gộp bảng khảo sát sau chạy bổ sung

Gộp bảng gốc đã audit và các run 18/19 đã hoàn tất. Mỗi loại chỉ chọn một run; chạy lại cùng identity sẽ resume. Nhóm meta được đối chiếu với CE batch256. Không xếp hạng chung các backbone thành một kết luận về phương pháp trọng số.

In [1]:
from pathlib import Path
import os
import sys

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if not (ROOT / "supplement" / "reporting.py").is_file():
    raise RuntimeError("Mở notebook từ BAF_project hoặc BAF_project/notebooks sau khi giải nén.")

sys.path.insert(0, str(ROOT / "supplement"))
import reporting

reporting.setup(ROOT)
print("Python:", sys.executable)
print("Project:", ROOT)
print("Supplement:", reporting.VERSION)


Python: d:\UIT\.venvs\ml-gpu\Scripts\python.exe
Project: D:\UIT\DS317\BAF_project
Supplement: advisor-20261030-v1


### Chọn run nếu có nhiều phiên bản

In [2]:
# None: tự chọn khi mỗi loại chỉ có một run đã hoàn thành.
CE_RUN = None
SAMPLING_RUN = None


### Gộp và xuất bảng

In [3]:
from pathlib import Path
import pandas as pd

frames = [pd.read_csv(ROOT / "results/advisor_step1/test_metrics_verified.csv")]
for kind, selected in [("ce", CE_RUN), ("sampling", SAMPLING_RUN)]:
    if selected:
        path = Path(selected) / "evaluation/test_metrics.csv"
    else:
        choices = list((ROOT / "results/advisor_runs").glob(kind + "_*/evaluation/test_metrics.csv"))
        if len(choices) != 1:
            raise ValueError(f"Chọn {kind.upper()}_RUN: tìm thấy {len(choices)} run hoàn tất")
        path = choices[0]
    frames.append(pd.read_csv(path))
combined = pd.concat(frames, ignore_index=True)
# Bản đối chiếu meta lấy cùng kết quả LRE/MWN, không train lại.
meta = combined[combined.method.isin(["lre", "mwn"])].copy()
meta["comparison_group"] = "meta_matched_batch256"
combined = pd.concat([combined, meta], ignore_index=True)
OUTPUT = ROOT / "results/advisor_combined"
OUTPUT.mkdir(parents=True, exist_ok=True)
combined.to_csv(OUTPUT / "test_metrics_all.csv", index=False)
summary = reporting.summarize(combined, OUTPUT)
print(summary.to_string(index=False))
print("ADVISOR_COMBINED_READY", OUTPUT)
print(reporting.export_proof(OUTPUT))

TABLE_READY 27 methods 81 seed records
           suite          comparison_group            method  ap_mean   ap_std  roc_auc_mean  roc_auc_std  recall_at_fpr5_curve_mean  recall_at_fpr5_curve_std  recall_frozen_threshold_mean  recall_frozen_threshold_std  fpr_frozen_threshold_mean  fpr_frozen_threshold_std  precision_frozen_threshold_mean  precision_frozen_threshold_std  alert_rate_frozen_threshold_mean  alert_rate_frozen_threshold_std  older_fpr_mean  older_fpr_std  younger_fpr_mean  younger_fpr_std  older_tpr_mean  older_tpr_std  younger_tpr_mean  younger_tpr_std  fpr_gap_older_minus_younger_mean  fpr_gap_older_minus_younger_std  fpr_ratio_min_max_mean  fpr_ratio_min_max_std  fit_seconds_mean  fit_seconds_std  n_seeds
      advisor_ce     meta_matched_batch256       ce_batch256 0.196808 0.006427      0.887373     0.002499                   0.540149                  0.008756                      0.532913                     0.010316                   0.048389                  0.0041